In [1]:
from qutip import *
import numpy as np
import matplotlib.pyplot as plt

In [2]:
def D(alpha,n):
    return tensor([identity(2)]*n +[displace(cutoff,alpha)])

def F(n):
    F = (1j*np.pi/2*num(cutoff)).expm()
    return tensor([identity(2)]*n +[F])

In [3]:
def snot(N=None, target=0):
    if N is not None:
        return gate_expand_1toN(snot(), N, target)
    else:
        return 1 / np.sqrt(2.0) * Qobj([[1, 1],
                                        [1, -1]])

In [4]:
def qft(N=1):
    if N < 1:
        raise ValueError("Minimum value of N can be 1")

    N2 = 2 ** N
    phase = 2.0j * np.pi / N2
    arr = np.arange(N2)
    L, M = np.meshgrid(arr, arr)
    L = phase * (L * M)
    L = np.exp(L)
    dims = [[2] * N, [2] * N]
    return Qobj(1.0 / np.sqrt(N2) * L, dims=dims)


In [5]:
def Vj(lmbda,j,n): 
    qubit_pauli = tensor([identity(2)]*(j-1) + [sigmay()] + [identity(2)]*(n-j))
    # print(qubit_pauli)
    return tensor(qubit_pauli, (1j*np.pi)/(2**(j+1)*lmbda)*(destroy(cutoff) + create(cutoff))/np.sqrt(2.0)).expm()

def Wj(lmbda,j,n):
    qubit_pauli = tensor([identity(2)]*(j-1) + [sigmax()] + [identity(2)]*(n-j))
    if j == n:      
        disp_amount = -lmbda*2**(j-1)
    else:
        disp_amount = lmbda*2**(j-1)
    # print(qubit_pauli)
    return tensor(qubit_pauli, disp_amount*(destroy(cutoff) - create(cutoff))/np.sqrt(2.0)).expm()

def dv2cv_st_non_abelian(lmbda,n):
    U = tensor([identity(2)]*n + [identity(cutoff)])
    for j in range(n,0,-1):
        # print(j)
        # U = Wj(lmbda,j,n).dag() * Vj(lmbda,j,n).dag() * U
        U = Vj(lmbda,j,n).dag() * Wj(lmbda,j,n).dag() * U

    return U

def sinc2fock(delta):
    coeff = []
    return coeff

def dv_basis_state(s,n):
    
    gammas = 0.0
    if len(s)>2:
        for i in range(len(s)-1):
            gammas += (s[i] + s[i+1]) / 2.0
        gammas += (s[n-2] - s[n-1]) / 2.0
    else:
        gammas += (s[0] - s[1]) / 2.0
    # print(gammas)
    
    state = (-1)**gammas * tensor([(basis(2,0) + (-1)**((-i+1)/2) * basis(2,1)) for i in s])
    
    return state.unit()

In [6]:
quantum_state = [[1,1,1,1],['00','01','10','11']]
# quantum_state = [[1,1],['0','1']]

In [7]:
psi_qubit = 0
for i in range(0,len(quantum_state[0])):
            psi_qubit += quantum_state[0][i]*tensor([basis(2,int(i)) for i in quantum_state[1][i][::-1]])
print(psi_qubit)

Quantum object: dims=[[2, 2], [1, 1]], shape=(4, 1), type='ket', dtype=Dense
Qobj data =
[[1.]
 [1.]
 [1.]
 [1.]]


In [9]:
cutoff_arr = [5]

allfidelity = np.zeros((len(cutoff_arr)))

hadamard = snot()

for c in range(0,len(cutoff_arr)):
    cutoff = 2**cutoff_arr[c]
    print("-----------------------")
    print('CUTOFF: 2**' + str(cutoff_arr[c]) )
    print("-----------------------")

    a = destroy(cutoff)
    x = (a + a.dag())/np.sqrt(2)

    # Initialize starting state
    delta = 18/(cutoff)
    delta_prime = 2*np.pi/(2**5 * delta)
    append_qubits = 1 
    Ust = dv2cv_st_non_abelian(delta,5)
    Ust1 = dv2cv_st_non_abelian(delta_prime,5).dag()
    num_ancillas = 1
    num_data_qubits = len(quantum_state[1][0])
    num_qubits = num_data_qubits + num_ancillas + 2*append_qubits # add data qubits and num_ancillas (and appended qubits)
    qubit_coeff = 2**num_qubits * [1] # uniform initial state
    qubit_init = tensor([snot()*basis(2,0), snot()*basis(2,0), snot()*basis(2,0), basis(2,0),basis(2,0)])

    print(num_qubits)

    # Parameters
    lmbda = 0.29 # 0.29 
    sr = np.log(1.12/lmbda)
    osc_init = basis(cutoff,0)


    psi0 = tensor(qubit_init ,osc_init)
    psi0 = psi0.unit()
    print("Initialization done")
    # print(psi0.unit())
    print(psi0.ptrace(0))
    # print(psi0)

    # basis transformation T^\dagger
    basis_trans_U = tensor([hadamard*sigmax()*sigmaz(), hadamard*sigmax(), hadamard*sigmax(), hadamard*sigmax(), hadamard*sigmaz()])
    psi_u = (tensor([basis_trans_U.dag(),identity(cutoff)]) * psi0)
    print("Basis transformation done")

    # state transfer unitary
    psi_u1 = Ust*psi_u
    print("DV to CV state transfer done")

    # Displacement Gate
    alpha = delta/2
    Dsym = D(alpha,num_qubits)

    psi_u2  = Dsym*psi_u1
    print("First displacement gate done")

    # QFT
    QFT     = F(num_qubits)
    psi_QFT = QFT*psi_u2
    print("QFT gate done")

    # Displacement gate
    alpha = -delta_prime/2
    Dsym1 = D(alpha,num_qubits)
    psi_u3  = Dsym1*psi_QFT
    print("Second displacement gate done")

    # state transfer unitary
    psi_u4 = Ust1*psi_u3
    print("CV to DV state transfer done")

    psi_u5 = (tensor([basis_trans_U,identity(cutoff)]) * psi_u4)
    print("Final Basis transformation done")
    # print(psi_u5)

    psi_final       = (psi_u5).unit()
    rho_cav_final   = (psi_final.ptrace([num_qubits])).unit()
    rho_qubit_final = (psi_final.ptrace([2,3])).unit()

    Uexact   = qft(num_data_qubits)
    qubitQFT = (Uexact.unit()*psi_qubit.unit())
    print(qubitQFT.unit())
    allfidelity[c]   = fidelity(rho_qubit_final.unit(),qubitQFT.unit())

    print('QFT Fidelity: ' + str(allfidelity[c])) # Fidelity with exact QFT
    print('-------------------------')






-----------------------
CUTOFF: 2**5
-----------------------
5
Initialization done
Quantum object: dims=[[2], [2]], shape=(2, 2), type='oper', dtype=Dense, isherm=True
Qobj data =
[[0.5 0.5]
 [0.5 0.5]]
Basis transformation done
DV to CV state transfer done
First displacement gate done
QFT gate done
Second displacement gate done
CV to DV state transfer done
Final Basis transformation done
Quantum object: dims=[[2, 2], [1, 1]], shape=(4, 1), type='ket', dtype=Dense
Qobj data =
[[ 1.00000000e+00]
 [-3.06161700e-17]
 [ 0.00000000e+00]
 [ 9.18485099e-17]]
QFT Fidelity: 0.8339489166299793
-------------------------


In [10]:
print(rho_qubit_final)

Quantum object: dims=[[2, 2], [2, 2]], shape=(4, 4), type='oper', dtype=Dense, isherm=True
Qobj data =
[[ 0.6954708 +0.j          0.03290058+0.01406572j -0.01049622+0.01242078j
   0.0011555 +0.00482412j]
 [ 0.03290058-0.01406572j  0.05727319+0.j         -0.01298055-0.01287999j
  -0.01286652-0.00387944j]
 [-0.01049622-0.01242078j -0.01298055+0.01287999j  0.12522187+0.j
   0.00658317+0.01521327j]
 [ 0.0011555 -0.00482412j -0.01286652+0.00387944j  0.00658317-0.01521327j
   0.12203414+0.j        ]]
